# Apache Hudi

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook)*



*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

Apache Hudi (Hadoop Upserts Deletes and Incrementals) is an open table format and lakehouse framework that brings database-like primitives to object storage: record-level upserts, deletes, incremental pulls, and pluggable table services. It was created at Uber in 2016 to keep petabyte-scale trip tables fresh without rebuilding partitions.

## What is Apache Hudi?

Apache Hudi is an open-source data lakehouse framework that brings database-like capabilities (upserts, deletes, incremental pulls) to data lakes on object storage. It manages table metadata and timelines so batch and streaming engines can write and read consistently. Hudi is not a query engine: Spark, Flink, Trino, Presto, Hive, and BigQuery-style external readers use Hudi metadata to read the same table.

Hudi's differentiators versus a plain Parquet dataset:

+ A **timeline** in `.hoodie/` records every instant (commit, deltacommit, compaction, clean, replacecommit) and lets readers see a consistent snapshot.
+ A **record key index** lets an upsert locate the file group holding a key instead of rewriting the whole table.
+ **Table services** (compaction, clustering, cleaning, archival) run inline or asynchronously to keep the table healthy.
+ **Incremental queries** return only records changed since an instant, which enables streaming-on-batch pipelines.

```text
Spark / Flink / DeltaStreamer
        |
        v
s3://lake/hudi/orders/
  .hoodie/                 table metadata + timeline
    hoodie.properties
    metadata/              metadata table (files + column stats index)
    20261004013000.commit    completed instant
    20261004020000.deltacommit
  order_date=2026-10-04/   partitions
    <fileId>_<token>_<instant>.parquet     base files
```

A first write and read in PySpark:

In [1]:
try:
    orders_df, spark
except NameError:
    print("SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.")
else:
    hudi_options = {
        "hoodie.table.name": "orders_hudi",
        "hoodie.datasource.write.recordkey.field": "order_id",
        "hoodie.datasource.write.partitionpath.field": "order_date",
        "hoodie.datasource.write.precombine.field": "updated_at",
        "hoodie.datasource.write.table.type": "COPY_ON_WRITE",
        "hoodie.datasource.write.operation": "upsert",
        "hoodie.datasource.write.keygenerator.class": "org.apache.hudi.keygen.SimpleKeyGenerator",
    }

    (
        orders_df.write.format("hudi")
        .options(**hudi_options)
        .mode("append")
        .save("s3://lake/hudi/orders")
    )

    orders = spark.read.format("hudi").load("s3://lake/hudi/orders")


SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.


Interview framing: Hudi is the format you reach for when the dominant workload is continuous upsert/delete ingestion, when you need incremental pull as a first-class read mode, and when you want table services managed inside the write path.

How a Hudi table is organized under the hood. Every write creates an instant on the timeline. An instant moves through `requested`, `inflight`, and `completed`. Readers only see completed instants, which is what gives snapshot isolation. Data lives in file groups. A file group is one `fileId` in one partition. In a COW table a file group is a stack of base file versions. In a MOR table it is a base file plus one or more log files. The set of files a reader needs at one instant is a file slice: the latest base file plus the log files newer than it.

Worked example. A 500 GB COW orders table has about 4000 file groups of 128 MB each. A batch updates 1% of the keys, roughly 40 file groups. Hudi reads those 40 base files, merges the new rows, and writes 40 new base files. One commit instant appears on the timeline. The other 3960 file groups are untouched. A reader that starts after the commit sees the new files; a reader that started before it keeps reading the old slice.

Common mistake: treating the table directory as a plain Parquet dataset. Copying, renaming, or deleting files by hand breaks the timeline and can leave readers pointing at a file slice that no longer exists. Go through Hudi writers and table services for every change.

## What problems does Hudi solve in a data lake?

Hudi enables efficient updates and deletes without rewriting entire datasets, supports incremental processing (read only changes), and helps manage small files and write amplification. It is commonly used for near-real-time ingestion and CDC-style workloads where tables must be kept up-to-date.

Concretely, Hudi solves five recurring data-lake problems:

+ **Updating a small fraction of a large table.** Rewriting 50 TB of Parquet to change 1% of rows is wasteful. Hudi rewrites only the affected file groups.
+ **Deletes and GDPR erasure.** Path-based lakes cannot delete a single customer's rows cheaply. Hudi supports record-level deletes and a delete-marker payload for soft deletes.
+ **Incremental processing.** Extracting "what changed since the last run" from Parquet means scanning and diffing. Hudi's incremental query returns exactly the changed records since an instant.
+ **Small files and write amplification.** Frequent micro-batch ingestion creates thousands of tiny files. Hudi's file-sizing, cleaning, and clustering services bound file counts and file sizes.
+ **Consistency for concurrent readers and writers.** A commit either fully succeeds or is rolled back; readers see the last successful instant even while writers are appending.

Non-goals worth stating in an interview: Hudi is not a compute engine, not a replacement for a warehouse's SQL optimizer, and not a low-latency point-lookup store. It optimizes lake-scale analytical ingestion, not OLTP.

Mechanics of a safe change. Hudi writes the new files first, then commits an instant. The commit is the atomic swap. If the writer dies half way, no completed instant references the partial files and a rollback deletes them. This is multi-version concurrency control: file versions accumulate, a reader pins one version, and cleaning later removes versions that no reader or incremental consumer still needs.

Worked example, incremental versus full scan. A 10 TB table receives 5 GB of changes per hour. A full scan reads 10 TB per run. An incremental pull reads only the records committed after the last checkpoint, about 5 GB. Over a day that is 240 TB scanned versus 120 GB. That gap is the reason incremental queries exist, and it is why the checkpoint instant must be stored reliably.

Common mistake: using Hudi as a point-lookup store. An upsert indexed by key still reads and rewrites whole base files. A request for one row by primary key at millisecond latency belongs in a database or KV store, not in object storage.

## What are Hudi's table types and physical file layout?

Hudi has two table types, `COPY_ON_WRITE` (COW) and `MERGE_ON_READ` (MOR), and a file-group abstraction shared by both.

+ A **file group** is identified by a stable `fileId` and lives in one partition. All versions of a record's data land in the same file group, which is what makes the index lookup cheap.
+ A **base file** is a columnar file, usually Parquet, named `<fileId>_<writeToken>_<instantTime>.parquet`.
+ A **log file** (MOR only) is an Avro-encoded delta log named `.<fileId>_<baseInstantTime>.log.<version>_<writeToken>`; it holds changes until compaction folds them into a new base file.
+ The **`.hoodie` directory** holds `hoodie.properties` (table config), the active timeline, archived timeline, and the optional metadata table.

```text
s3://lake/hudi/orders/
  .hoodie/
    hoodie.properties
    20261004013000.commit
    20261004020000.commit.requested
    metadata/
      files/            file listing index
      column_stats/     column statistics index
  order_date=2026-10-04/
    7f3a2c1b-...-0_20261004013000.parquet
    .7f3a2c1b-...-0_20261004013000.log.1_20261004020000
```

Timeline actions you should be able to name:

| Action | Meaning |
|---|---|
| `commit` | A completed COW write |
| `deltacommit` | A completed MOR write that appended log files |
| `replacecommit` | Atomically replaces file groups; used by clustering and insert-overwrite |
| `compaction` | Folds MOR log files into new base files |
| `clean` | Deletes old file versions and log files beyond retention |
| `rollback` | Removes a failed or aborted instant's files |
| `savepoint` | Pins a file version so cleaning cannot remove it |

Inspect the layout from Spark:

In [2]:
try:
    spark
except NameError:
    print("SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.")
else:
    meta = spark.read.format("hudi").load("s3://lake/hudi/orders")
    meta.select("_hoodie_commit_time", "_hoodie_file_name", "_hoodie_partition_path").show(5, truncate=False)


SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.


Every Hudi record carries hidden metadata columns: `_hoodie_commit_time`, `_hoodie_commit_seqno`, `_hoodie_record_key`, `_hoodie_partition_path`, and `_hoodie_file_name`. They are how Hudi tracks record provenance per row.

File slices and versions. Compaction produces a new base file and sets a new base instant for the file group. The previous slice stays on disk until cleaning removes it, so time travel, rollbacks, and long-running reads still work after a compact. A file group can be seen as a chain of file slices ordered by instant time.

Timeline states. An instant starts as `<instant>.requested`, becomes `<instant>.inflight` while a writer holds it, and finishes as `<instant>.<action>` when complete. The action is one of commit, deltacommit, replacecommit, compaction, clean, rollback, or savepoint.

Worked example, one file group over three writes:

```text
t1  deltacommit   base parquet written for fileId 7f3a
t2  deltacommit   .7f3a.log.1 appended (two updates)
t3  deltacommit   .7f3a.log.2 appended (one delete)
t4  compaction    new base parquet from base + log.1 + log.2
t5  clean         base(t1) and log files retired from disk
```

Common mistake: reading the instant in a file name as the commit time of the rows inside it. The file name instant says when the file was written. The per-row `_hoodie_commit_time` says when that row version landed, and it is the column to trust for provenance.

## How does Hudi handle upserts and deletes?

Hudi manages record-level changes on a data lake table using a record key, precombine field, and table timeline. It can apply inserts, updates, and deletes efficiently compared with rewriting an entire dataset.

The mechanics of one upsert batch:

1. Hudi reads the input records, computes each record key, and resolves duplicates using the precombine field.
2. The index maps each key to an existing file group, or tags the record as an insert for a new file group.
3. Hudi groups records by file group, reads only those base files (plus log files on MOR), merges the new records, and writes the result.
4. A new instant is committed to the timeline; readers atomically switch to the new snapshot.

Deletes come in two flavours:

+ **Hard delete**: the record is physically removed via the `delete` operation, which targets the file groups holding the keys.
+ **Soft delete**: the record is written with a null ordering value (or a delete marker), so readers filter it out while the row remains for audit. Hudi's `DefaultHoodieRecordPayload` treats a record whose precombine value is null as a delete.

In [3]:
try:
    spark
except NameError:
    print("SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.")
else:
    # Hard delete of two orders
    spark.createDataFrame([("order-1001",), ("order-1002",)], ["order_id"]) \
        .write.format("hudi") \
        .option("hoodie.table.name", "orders_hudi") \
        .option("hoodie.datasource.write.recordkey.field", "order_id") \
        .option("hoodie.datasource.write.partitionpath.field", "order_date") \
        .option("hoodie.datasource.write.operation", "delete") \
        .option("hoodie.datasource.write.keygenerator.class", "org.apache.hudi.keygen.NonpartitionedKeyGenerator") \
        .mode("append") \
        .save("s3://lake/hudi/orders_nopartition")


SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.


Failure modes to mention: late-arriving events with an older precombine value are ignored (correctly), but events without a reliable ordering column produce non-deterministic results. Duplicate keys inside one batch are resolved by precombine, so choose a column that is monotonic and never null for real updates.

The merge step. The index tags every incoming record with the file group that owns its key, or marks it as an insert when no file group is found. Records are grouped by file group and merged. On COW the merge reads the base file, applies the new records, and writes a new base file. On MOR the merge appends a log block and returns quickly. Duplicate keys inside the same batch are collapsed first, using the precombine field.

Worked delete example. To hard delete order 1001, write a frame that contains only the key and use `hoodie.datasource.write.operation=delete`. To soft delete with the default payload, upsert the row with a null precombine value instead. Readers then treat the row as gone while the version history still shows what happened.

Common mistake: a soft delete that nulls a data column but keeps a non-null ordering value. The row stays live and the delete silently does nothing. Null the precombine field, or switch to a delete-marker payload so the intent is explicit.

## How does Hudi support upserts?

Hudi indexes records by key and can locate existing records to update them. On upsert, it writes new versions and updates the timeline/metadata so readers see a consistent view. Correctness depends on stable keys and proper handling of late and duplicate events.

The value of the index is asymmetric cost: without it, Spark would have to shuffle the entire table against the incoming batch. With it, Hudi reads only the file groups that contain touched keys.

```text
incoming batch (1M changed orders)
    -> key generation (order_id)
    -> index lookup (bloom filter / HFile / bucket)
    -> touched file groups (2,000 of 500,000)
    -> merge + write 2,000 new base files
```

Guidance for reliable upserts:

+ Choose a stable, immutable record key. If the key changes when the business entity changes, you get duplicates instead of updates.
+ Provide a precombine field that reflects event time (`updated_at`, `source_lsn`) rather than ingestion time; otherwise replays and late events win incorrectly.
+ Keep record keys evenly distributed so the shuffle and the index do not create a single hot task.
+ Control the `hoodie.upsert.shuffle.parallelism` to roughly match data volume; too few tasks cause long, memory-heavy merges.
+ Use `bulk_insert` for the initial historical load and `upsert` only for incremental changes; bulk insert skips the index and is far faster for empty tables.

The same knobs from the Java/Scala write client, which is what the datasource builds for you:

```scala
import org.apache.hudi.config.HoodieWriteConfig
import org.apache.hudi.config.HoodieIndexConfig
import org.apache.hudi.index.HoodieIndex

val cfg = HoodieWriteConfig.newBuilder()
  .withPath("s3://lake/hudi/orders")
  .withSchema(schemaString)
  .withParallelism(200, 100)          // (insert, upsert) shuffle parallelism
  .forTable("orders_hudi")
  .withIndexConfig(
    HoodieIndexConfig.newBuilder()
      .withIndexType(HoodieIndex.IndexType.BLOOM)
      .bloomIndexParallelism(100)
      .build()
  )
  .build()
```

The option names map onto the builder. `withParallelism(insert, upsert)` is `hoodie.insert.shuffle.parallelism` and `hoodie.upsert.shuffle.parallelism`. The index builder is `hoodie.index.type` plus its tuning options. The datasource collects the option map and constructs this same `HoodieWriteConfig` internally.

Common mistake: a record key that moves partitions. A local index cannot find the old file group, so the old copy survives as a stale duplicate. Fix the key generator so the partition path is stable per key, or use a global index if the move is unavoidable.

## What is a record key, partition path, and precombine field?

Record key identifies a row uniquely (like a primary key). Partition path controls how data is grouped for storage and pruning. Precombine field is used to resolve multiple records for the same key in a batch (for example choose the latest by `updated_at`) to ensure deterministic upserts.

```text
record key: order_id
precombine field: updated_at
partition path: order_date
```

In [4]:
{
    "hoodie.datasource.write.recordkey.field": "order_id",
    "hoodie.datasource.write.precombine.field": "updated_at",
    "hoodie.datasource.write.partitionpath.field": "order_date",
    "hoodie.datasource.write.keygenerator.class": "org.apache.hudi.keygen.SimpleKeyGenerator",
}

{'hoodie.datasource.write.recordkey.field': 'order_id',
 'hoodie.datasource.write.precombine.field': 'updated_at',
 'hoodie.datasource.write.partitionpath.field': 'order_date',
 'hoodie.datasource.write.keygenerator.class': 'org.apache.hudi.keygen.SimpleKeyGenerator'}

Practical rules:

+ The partition path must be derivable from the record or supplied as a literal string. A partition column with thousands of values (for example `order_id`) creates tiny partitions and destroys file sizing.
+ Composite record keys are supported with `ComplexKeyGenerator` and a comma-separated list, for example `"tenant_id,order_id"`.
+ Timestamp-derived partitions use `TimestampBasedKeyGenerator` with `hoodie.keygen.timebased.timestamp.type` and format options.
+ The precombine comparison is per key. Ties (equal ordering values) are resolved deterministically by the payload, but you should not depend on arbitrary tie-breaking; make the ordering value unique per event where possible.
+ Non-partitioned tables use `NonpartitionedKeyGenerator` and live entirely at the table root, which is fine for small tables but loses pruning on large ones.

Worked example. A orders table uses `order_id` as the record key, `order_date` as the partition path, and `updated_at` as the precombine field. Two events for order 1001 arrive in one batch, at 10:00 and 10:05. Hudi keeps the 10:05 record and discards the earlier one. A third event arrives an hour later with `updated_at` 09:30, from a delayed shard. It loses the comparison and is ignored, which is the correct behaviour for an event-time ordering.

Composite keys. `"org.apache.hudi.keygen.ComplexKeyGenerator"` with `hoodie.datasource.write.recordkey.field=tenant_id,order_id` makes the pair unique. The precombine comparison is still per full key, so tenant A and tenant B can hold the same `order_id` without interfering.

Common mistake: a precombine field that is ingestion time rather than event time. A replay or a backfill then overwrites good data with older records, and the table looks correct while the contents are wrong. Use a monotonic source field such as `source_lsn` or the business `updated_at`.

## How do you write and upsert data with Hudi in PySpark?

A Hudi write is a normal `DataFrameWriter` with Hudi options. The options decide table type, operation, key semantics, indexing, and table services. This is the example most interviews ask you to produce from memory.

In [5]:
try:
    orders_df
except NameError:
    print("SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.")
else:
    hudi_options = {
        "hoodie.table.name": "orders_hudi",
        "hoodie.datasource.write.recordkey.field": "order_id",
        "hoodie.datasource.write.partitionpath.field": "order_date",
        "hoodie.datasource.write.precombine.field": "updated_at",
        "hoodie.datasource.write.table.type": "MERGE_ON_READ",
        "hoodie.datasource.write.operation": "upsert",
        "hoodie.datasource.write.keygenerator.class": "org.apache.hudi.keygen.SimpleKeyGenerator",
        "hoodie.datasource.write.hive_style_partitioning": "true",
        "hoodie.index.type": "BLOOM",
        "hoodie.upsert.shuffle.parallelism": "200",
        "hoodie.parquet.max.file.size": "134217728",
        "hoodie.parquet.small.file.limit": "104857600",
        "hoodie.metadata.enable": "true",
        "hoodie.clean.automatic": "true",
        "hoodie.cleaner.policy": "KEEP_LATEST_COMMITS",
        "hoodie.cleaner.commits.retained": "10",
        "hoodie.compact.inline": "true",
        "hoodie.compact.inline.max.delta.commits": "5",
    }

    (
        orders_df.write.format("hudi")
        .options(**hudi_options)
        .mode("append")
        .save("s3://lake/hudi/orders")
    )


SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.


Initial historical load with bulk insert (no index lookup, fastest path into an empty table):

In [6]:
try:
    dict, history_df
except NameError:
    print("SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.")
else:
    bulk_options = dict(hudi_options, **{
        "hoodie.datasource.write.operation": "bulk_insert",
        "hoodie.bulkinsert.shuffle.parallelism": "400",
        "hoodie.bulkinsert.user.defined.partitioner.class": "org.apache.hudi.execution.bulkinsert.RDDCustomColumnsSortPartitioner",
        "hoodie.bulkinsert.user.defined.partitioner.sort.columns": "order_id",
    })

    history_df.write.format("hudi").options(**bulk_options).mode("overwrite").save("s3://lake/hudi/orders")


SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.


Insert-overwrite for a full partition replacement (for example reprocessing one business day):

In [7]:
try:
    corrected_day_df, dict
except NameError:
    print("SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.")
else:
    reprocess_options = dict(hudi_options, **{
        "hoodie.datasource.write.operation": "insert_overwrite",
        "hoodie.datasource.write.partitionpath.field": "order_date",
    })

    (
        corrected_day_df.write.format("hudi")
        .options(**reprocess_options)
        .mode("append")
        .save("s3://lake/hudi/orders")
    )


SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.


Interview notes:

+ `operation=upsert` with `mode("append")` is the normal incremental path; `mode("overwrite")` on an existing table is usually wrong and can wipe history.
+ `hoodie.datasource.write.table.type` is fixed when the table is created. Converting COW to MOR requires a rewrite or a migration job.
+ In Databricks you can register the Hudi table in Unity Catalog with `CREATE TABLE ... USING hudi LOCATION ...`, but check runtime support for the Hudi version you depend on.
+ Keep the Hudi bundle version aligned with the Spark version (`hudi-spark3.5-bundle_2.12` for Spark 3.5), because Hudi and Spark internals are tightly coupled.

Create the table and register it in Spark SQL, then write with SQL as well as the datasource:

```sql
-- Register an existing Hudi path as a Spark SQL table.
CREATE TABLE orders_hudi
USING hudi
LOCATION 's3://lake/hudi/orders';
```

MERGE INTO on a Hudi table works through the Hudi Spark SQL support and avoids a hand-rolled upsert:

```sql
MERGE INTO orders_hudi AS t
USING staged_orders AS s
ON t.order_id = s.order_id
WHEN MATCHED AND s.op = 'DELETE' THEN DELETE
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
```

Choosing the operation:

+ `upsert` looks up every key in the index and merges. Use it for incremental changes.
+ `insert` writes new rows without merging and keeps small-file handling. Use it for an append-only feed that is known to be duplicate-free.
+ `bulk_insert` skips the index entirely and bin-packs records into files. Use it for the initial load of an empty table, never on an existing table.

In [8]:
try:
    append_only_df, dict
except NameError:
    print("SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.")
else:
    # Insert-only feed, then a bulk insert for a fresh historical load.
    insert_options = dict(hudi_options, **{
        "hoodie.datasource.write.operation": "insert",
        "hoodie.datasource.insert.dup.policy": "drop",
    })

    (
        append_only_df.write.format("hudi")
        .options(**insert_options)
        .mode("append")
        .save("s3://lake/hudi/orders")
    )


SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.


Common mistake: `mode("overwrite")` with `operation=upsert` on an existing table. It can drop history instead of merging. Incremental writes use `mode("append")`; only `insert_overwrite` is meant to replace a partition, and it does so through a `replacecommit`.

## What indexes does Hudi use to find records quickly?

Hudi's index maps record keys to file groups so an upsert only touches affected files. The index choice is one of the biggest levers on upsert latency and write amplification.

| Index | How it works | Best for |
|---|---|---|
| `BLOOM` (default) | Bloom filter per base file, probed by the incoming keys | Large partitioned tables, most upsert workloads |
| `SIMPLE` | Keeps a full key-to-file map in memory | Small tables that fit in executor memory |
| `GLOBAL_BLOOM` | Bloom filter across all partitions | Global uniqueness when keys can move partitions |
| `HFILE` | Stores the key map in an HFile per file group | Large tables where a persistent, fast lookup beats recomputing blooms |
| `BUCKET` | Hash of the key determines the file group; no lookup at all | Streaming ingestion with a fixed bucket count |

In [9]:
{
    "hoodie.index.type": "BLOOM",
    "hoodie.bloom.index.parallelism": "100",
    "hoodie.bloom.index.prune.by.runtime.filter": "true",
    "hoodie.bloom.index.update.partition.path": "true",
    "hoodie.metadata.enable": "true",
}

{'hoodie.index.type': 'BLOOM',
 'hoodie.bloom.index.parallelism': '100',
 'hoodie.bloom.index.prune.by.runtime.filter': 'true',
 'hoodie.bloom.index.update.partition.path': 'true',
 'hoodie.metadata.enable': 'true'}

In [10]:
# Bucket index removes the lookup entirely but fixes the file layout at creation time.
{
    "hoodie.index.type": "BUCKET",
    "hoodie.index.bucket.engine": "CONSISTENT_HASHING",
    "hoodie.bucket.index.num.buckets": "64",
    "hoodie.storage.layout.partitioner.class": "org.apache.hudi.table.action.commit.SparkBucketIndexPartitioner",
}

{'hoodie.index.type': 'BUCKET',
 'hoodie.index.bucket.engine': 'CONSISTENT_HASHING',
 'hoodie.bucket.index.num.buckets': '64',
 'hoodie.storage.layout.partitioner.class': 'org.apache.hudi.table.action.commit.SparkBucketIndexPartitioner'}

Trade-offs interviewers look for:

+ Bloom indexes trade memory and CPU on every upsert for flexibility; bucket indexes trade flexibility for near-zero lookup cost.
+ `BLOOM` can produce false positives, which only cause an unnecessary file read, never a wrong result.
+ Global indexes cost more because the key space spans partitions; use local indexes unless keys genuinely move between partitions.
+ The metadata table (`hoodie.metadata.enable=true`) stores the files index and column statistics index, which speeds up query planning by avoiding repeated object-store listing. It is effectively mandatory on cloud object storage.

How the lookup works. A Bloom index keeps a small filter per base file. Hudi probes the filters with the incoming keys and reads only the files whose filter may match. A false positive causes one extra file read, never a wrong answer. An HFile index stores the key-to-fileId mapping in a sorted, immutable HFile per file group, which avoids rebuilding filters on every upsert. A simple index keeps the whole key map in memory on the driver, which is fast and fragile once the table grows. A bucket index hashes the key to a bucket, so there is no lookup at all.

Worked example. A 1M-row batch upserts into a table of 500,000 file groups. With a working index, the batch touches about 2,000 groups, so 2,000 base files are read and rewritten. Without an index, Spark shuffles and rewrites the whole table. The index is the difference between a few minutes and many hours.

Common mistake: reaching for a global index on a very large table by default. A global index widens the lookup across partitions and costs more memory and CPU. Use a local index unless keys genuinely migrate between partitions.

## What is the difference between Copy-on-Write (COW) and Merge-on-Read (MOR)?

COW stores data in columnar files and rewrites them on updates, optimizing read performance at the cost of write amplification. MOR stores base files plus delta log files and merges them at read time (or during compaction), optimizing write performance at the cost of more complex reads.

| Dimension | COW | MOR |
|---|---|---|
| Upsert write cost | Rewrites affected base files | Appends small log files |
| Read cost (snapshot) | Low, pure columnar | Higher, merges base + logs |
| Read cost (read_optimized) | Same as snapshot | Ignores logs, may be stale |
| Storage overhead | Multiple file versions until cleaning | Versions plus log files |
| Compaction requirement | None | Essential; backlog degrades reads |
| Streaming ingestion | Higher latency | Low latency |
| Engine compatibility | Broadest | Requires Hudi-aware readers for logs |
| Typical partition size | Moderate to large | Large, write-heavy |

Config snapshots of each table type:

In [11]:
cow = {
    "hoodie.datasource.write.table.type": "COPY_ON_WRITE",
    "hoodie.datasource.write.operation": "upsert",
    "hoodie.parquet.max.file.size": "134217728",
}

mor = {
    "hoodie.datasource.write.table.type": "MERGE_ON_READ",
    "hoodie.datasource.write.operation": "upsert",
    "hoodie.compact.inline": "true",
    "hoodie.compact.inline.max.delta.commits": "5",
    "hoodie.logfile.max.size": "1073741824",
}

Interview trap: MOR with a large compaction backlog looks fast to write and terrible to read. Always pair MOR with an explicit compaction SLA and alerting on delta-commit lag.

Worked cost example. A 2 TB COW table receives an upsert that touches 1% of a day's partition, about 20 GB of base files. COW reads and rewrites 20 GB, roughly 40 GB of I/O. The same change on MOR appends a few GB of log files, but a snapshot read now merges those logs and a read-optimized read misses them until compaction. After compaction the base files are clean again and the snapshot read is fast.

Which engines can read each type. COW exposes plain Parquet files plus Hudi metadata, so almost any Parquet reader can be pointed at it. MOR snapshot reads need a Hudi-aware reader because the log files are Avro-encoded. Verify engine support before choosing MOR.

Common mistake: switching a table type in place. `hoodie.datasource.write.table.type` is fixed when the table is created. Converting COW to MOR, or the reverse, needs a rewrite or a migration job, not an option change on the next write.

## How do you choose between COW and MOR?

Choose COW when read latency and query simplicity are most important and updates are moderate. Choose MOR when write throughput and near-real-time updates are critical, and you can manage compaction and read-time merges. The choice also depends on engine support and operational constraints.

A decision checklist:

+ If updates touch a small fraction of rows and reads dominate, COW is usually cheaper end to end.
+ If ingestion is a continuous stream and a 5-15 minute freshness SLA matters, MOR ingests faster and compacts on a schedule.
+ If downstream engines cannot read Hudi log files, COW avoids a whole class of compatibility problems.
+ If write amplification is the bottleneck (network or object-store PUT limits), MOR is the escape valve.
+ If your team cannot operate a compaction job reliably, prefer COW; an unmanaged MOR table degrades continuously.

In [12]:
# Heuristic: pick MOR for streaming, COW for batch + BI.
def pick_table_type(rows_per_hour: int, readers: int, freshness_minutes: int) -> str:
    if freshness_minutes <= 15 and readers < 5:
        return "MERGE_ON_READ"
    return "COPY_ON_WRITE"


print(pick_table_type(rows_per_hour=2_000_000, readers=1, freshness_minutes=5))
print(pick_table_type(rows_per_hour=50_000, readers=40, freshness_minutes=60))


MERGE_ON_READ
COPY_ON_WRITE


Cost note: MOR reduces write cost but can increase read cost and storage until cleaning runs. The honest answer is often "COW for the curated BI table, MOR for the bronze ingestion table."

Worked decision. A BI table with 40 dashboards, 5% of rows updated per day, and a one-hour freshness target is COW: reads dominate and updates are cheap to absorb. A CDC bronze table with 15,000 events per second, a five-minute freshness target, and one downstream consumer is MOR: write throughput dominates and the single consumer can tolerate a read-time merge between compactions.

A numeric rule of thumb. If the fraction of rows rewritten per run is under a few percent and reads outnumber writes, COW usually wins end to end. If updates are continuous and the write path is the bottleneck, MOR wins on ingestion but you must budget for compaction CPU and storage.

Common mistake: picking MOR for a table with no compaction schedule and no owner. The write path looks great for weeks, then snapshot reads slow down and the incident is on the read side. Table services are part of the design, not an afterthought.

## What are the Hudi query types and how do you run an incremental pull?

Hudi exposes three query types from Spark, selected with `hoodie.datasource.query.type`.

+ `snapshot` (default): the latest committed view. On MOR it merges base files with log files at read time.
+ `read_optimized`: base files only. On COW it is identical to snapshot; on MOR it returns the last compacted state and can be stale.
+ `incremental`: only records committed in a given instant range, which powers incremental pipelines and CDC sinks.

In [13]:
try:
    spark
except NameError:
    print("SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.")
else:
    # Snapshot read (latest data, merges MOR logs)
    snapshot_df = (
        spark.read.format("hudi")
        .option("hoodie.datasource.query.type", "snapshot")
        .load("s3://lake/hudi/orders")
    )

    # Read-optimized read (fast, base files only)
    ro_df = (
        spark.read.format("hudi")
        .option("hoodie.datasource.query.type", "read_optimized")
        .load("s3://lake/hudi/orders")
    )


SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.


Incremental pull with a saved checkpoint:

In [14]:
try:
    spark
except NameError:
    print("SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.")
else:
    begin = "20261004000000000"   # last processed Hudi instant
    end = "20261005000000000"     # inclusive upper bound; omit for "up to now"

    changes = (
        spark.read.format("hudi")
        .option("hoodie.datasource.query.type", "incremental")
        .option("hoodie.datasource.read.begin.instanttime", begin)
        .option("hoodie.datasource.read.end.instanttime", end)
        .option("hoodie.datasource.read.incr.fallback.fulltablescan.enable", "false")
        .load("s3://lake/hudi/orders")
    )

    changes.write.mode("append").parquet("s3://lake/silver/orders_changes/")


SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.


Notes:

+ Persist the last committed instant in a control table; Hudi instants are the checkpoint. Reusing the same begin instant replays changes, so downstream must be idempotent.
+ Incremental queries on MOR pick up records from log files, which is why they are the standard CDC fan-out mechanism.
+ `read_optimized` is the right choice for dashboards that tolerate staleness but need predictable latency.
+ `_hoodie_commit_time` on each row tells you when that record version was committed, useful for debugging replay.

How incremental read works. You give Hudi a begin instant and an optional end instant. Hudi scans the timeline and returns records whose commit time falls in that range, including records recovered from MOR log files. Add `hoodie.datasource.query.incremental.format=cdc` to get change data with the operation type instead of only the latest state.

Worked checkpoint example. A job reads incrementally from instant `20261004000000000` and writes the result. On success it stores the instant it just consumed in a control table. The next run starts from that stored instant. If the job fails before the control table update, the next run replays the same range, so the sink must be idempotent.

Common mistake: a hard-coded begin instant in the job. A replay then either reprocesses history forever or skips changes. Keep the checkpoint in a table, in the commit metadata, or in the streaming checkpoint directory, and never in code.

## What is compaction in Hudi?

Compaction merges delta log files into base columnar files (mostly relevant for MOR tables). It improves read performance and reduces the overhead of reading many log files. Compaction must be scheduled and monitored to avoid backlog.

Compaction has two phases: a **planning** phase that writes a `compaction.requested` instant, and an **execution** phase that writes new base files and commits a `commit` that retires the log files.

Inline compaction (simple, but competes with ingestion):

In [15]:
{
    "hoodie.compact.inline": "true",
    "hoodie.compact.inline.max.delta.commits": "5",
    "hoodie.compact.inline.trigger.strategy": "NUM_COMMITS",
}

{'hoodie.compact.inline': 'true',
 'hoodie.compact.inline.max.delta.commits': '5',
 'hoodie.compact.inline.trigger.strategy': 'NUM_COMMITS'}

Async or scheduled compaction from a separate job, using Spark SQL procedures:

In [16]:
try:
    spark
except NameError:
    print("SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.")
else:
    spark.sql("call run_compaction(op => 'schedule', table => 'orders_hudi')")
    spark.sql("call run_compaction(op => 'run', table => 'orders_hudi')")


SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.


Or schedule then run as a standalone Spark job:

```bash
spark-submit \
  --class org.apache.hudi.utilities.HoodieCompactor \
  --packages org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0 \
  --conf spark.serializer=org.apache.spark.serializer.KryoSerializer \
  /opt/hudi/hudi-utilities-bundle.jar \
  --base-path s3://lake/hudi/orders \
  --table-name orders_hudi \
  --instant-time 20261005000000000 \
  --parallelism 100 \
  --spark-memory 4g
```

Operational guidance:

+ Give compaction its own resources so a large compaction does not starve ingestion.
+ Choose the trigger by delta-commit count or time, and alert on `hoodie.compaction.lag` in instants.
+ Compaction reads log files and rewrites base files, so it is I/O heavy; run it off-peak for large tables.
+ A compaction that keeps failing means readers keep merging an ever-growing log set, which is the classic MOR read-latency incident.

The two phases in detail. Scheduling scans the affected partitions, picks file slices whose log files are large enough, and writes a `compaction.requested` instant that contains the plan. Execution reads that plan, merges the base file with its log files for each slice, and commits a completed compaction that retires the log files. Only the second phase is I/O heavy.

When it triggers. `hoodie.compact.inline.trigger.strategy` controls the trigger. `NUM_COMMITS` fires after N delta commits, `TIME_ELAPSED` after N seconds, `NUM_AND_TIME` needs both, and `NUM_OR_TIME` needs either. The commit count comes from `hoodie.compact.inline.max.delta.commits`. Recent Hudi versions default to async compaction, so set `hoodie.compact.inline=true` only when you want the writer to do the work.

Log compaction. For MOR tables, log compaction stitches several small log blocks into one larger block without rewriting the base file. It is triggered by `hoodie.log.compaction.blocks.threshold` and scheduled with `hoodie.log.compaction.inline`. It reduces the number of blocks a snapshot read must merge.

Common mistake: sizing compaction resources from the ingestion job. A large compaction can read many GB and starve ingestion if it shares the same pool. Run it as its own job with its own parallelism, and alert on the delta-commit lag rather than only on job failure.

## What is clustering in Hudi and when do you need it?

Clustering reorganizes data files to improve layout (file sizes, sorting, locality) and query performance. You use clustering to combat small files, improve pruning, or optimize for common access patterns (for example sorting by event time).

Clustering rewrites selected file groups into better-shaped ones and commits the swap atomically as a `replacecommit`, so readers never see a partial state.

In [17]:
{
    "hoodie.clustering.inline": "true",
    "hoodie.clustering.inline.max.commits": "4",
    "hoodie.clustering.plan.strategy.target.file.max.bytes": "1073741824",
    "hoodie.clustering.plan.strategy.small.file.limit": "314572800",
    "hoodie.clustering.plan.strategy.sort.columns": "order_date,updated_at",
}

{'hoodie.clustering.inline': 'true',
 'hoodie.clustering.inline.max.commits': '4',
 'hoodie.clustering.plan.strategy.target.file.max.bytes': '1073741824',
 'hoodie.clustering.plan.strategy.small.file.limit': '314572800',
 'hoodie.clustering.plan.strategy.sort.columns': 'order_date,updated_at'}

Scheduled clustering with procedures or the standalone job:

In [18]:
try:
    spark
except NameError:
    print("SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.")
else:
    spark.sql("call run_clustering(table => 'orders_hudi')")


SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.


```bash
spark-submit \
  --class org.apache.hudi.utilities.HoodieClusteringJob \
  --packages org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0 \
  /opt/hudi/hudi-utilities-bundle.jar \
  --base-path s3://lake/hudi/orders \
  --table-name orders_hudi \
  --mode scheduleAndExecute \
  --parallelism 50
```

When clustering pays off:

+ After many small streaming commits, to consolidate hundreds of tiny files.
+ To sort data by a high-selectivity column so Parquet min/max statistics prune more row groups.
+ To change file sizing as data volume grows, without rewriting the whole table.
+ To fix skewed partitions, though a better partition key is usually the real fix.

Trade-offs: clustering is expensive (it rewrites data and can be GBs per run), it creates a new file version that cleaning must eventually remove, and inline clustering adds latency to the ingestion job. Schedule it rather than running it inline on busy pipelines.

How clustering runs. A plan phase groups file slices that should be rewritten together and writes a `clustering.requested` instant. An execute phase rewrites those groups, optionally sorting by `hoodie.clustering.plan.strategy.sort.columns`, and commits the swap as a single `replacecommit`. Because the swap is atomic, a reader sees either the old layout or the new one, never a mix.

Target sizing. `hoodie.clustering.plan.strategy.small.file.limit` picks the files that are too small to keep, and `hoodie.clustering.plan.strategy.target.file.max.bytes` sets the size each new file should reach. The defaults are 300 MB and 1 GB.

Common mistake: running clustering inline on a busy streaming table. The rewrite competes with ingestion for the same executors and can double commit latency. Schedule clustering as a separate job on quiet partitions instead.

## Why does the small files problem happen and how do you mitigate it in Hudi?

It happens when writers commit frequently with small batches or when partitions are highly granular. Mitigations include tuning write parallelism and file sizing, using clustering/compaction, and adjusting partitioning strategy to reduce tiny partitions.

Root causes:

+ Micro-batch streaming commits every minute, each writing a few MB.
+ Too many shuffle partitions, so each Spark task writes a tiny file.
+ Over-partitioning, for example by `order_id` or by hour on low-volume data.
+ `bulk_insert` with parallelism far higher than the data volume.

File-sizing and parallelism options:

In [19]:
{
    "hoodie.parquet.max.file.size": "134217728",
    "hoodie.parquet.small.file.limit": "104857600",
    "hoodie.parquet.compression.ratio": "0.35",
    "hoodie.insert.shuffle.parallelism": "200",
    "hoodie.upsert.shuffle.parallelism": "200",
    "hoodie.bulkinsert.shuffle.parallelism": "200",
    "hoodie.copyonwrite.insert.split.size": "500000",
}

{'hoodie.parquet.max.file.size': '134217728',
 'hoodie.parquet.small.file.limit': '104857600',
 'hoodie.parquet.compression.ratio': '0.35',
 'hoodie.insert.shuffle.parallelism': '200',
 'hoodie.upsert.shuffle.parallelism': '200',
 'hoodie.bulkinsert.shuffle.parallelism': '200',
 'hoodie.copyonwrite.insert.split.size': '500000'}

Mitigation strategy, in order of preference:

1. Write fewer, larger commits: a 5-minute micro-batch of 500 MB is far healthier than a 1-minute batch of 100 MB.
2. Set `hoodie.parquet.small.file.limit` so Hudi merges new records into existing small files instead of creating new ones.
3. Right-size shuffle parallelism: target roughly `data_size / 128 MB` tasks, not a fixed 200.
4. Coarsen partitioning. Daily partitions with 2-5 GB each are usually better than hourly partitions with 100 MB each.
5. Run clustering to consolidate existing files and sort them for pruning.
6. Schedule cleaning so old file versions do not inflate file counts and storage.

Interview answer: prevention is better than cleanup. Choose sane partitioning, batch enough data per commit, and tune Spark writer parallelism before reaching for clustering.

Worked sizing example. A stream ingests 120 GB per day in five-minute micro-batches. Each batch is about 120 GB / 288 = 426 MB. With a 128 MB target file size (`hoodie.parquet.max.file.size=134217728`), that is about four files per batch, or roughly 1150 files per day. The same stream in one-minute batches produces 100 MB per commit, and the writer creates one new file per commit instead of filling existing ones, giving five times the file count. Batching is the cheapest fix.

How auto-sizing works. When `hoodie.parquet.small.file.limit` is set, Hudi treats base files below that size as candidates and packs new records into them before creating new files. On MOR, `hoodie.merge.small.file.group.candidates.limit` caps how many small groups are considered so an upsert does not rewrite too many files.

Common mistake: creating a table from a large bulk insert with a parallelism far above the data volume. Each task then writes its own tiny file. Set `hoodie.bulkinsert.shuffle.parallelism` to roughly `input_size / target_file_size` and sort the input so related records land together.

## What table services does Hudi run and how do you schedule them?

Hudi table services are background operations that keep the table healthy. They can run inline inside a write or asynchronously in separate jobs.

| Service | Purpose | Key config |
|---|---|---|
| Compaction | Fold MOR log files into base files | `hoodie.compact.inline`, `hoodie.compact.inline.max.delta.commits` |
| Clustering | Reorganize file sizes and sort order | `hoodie.clustering.inline`, `hoodie.clustering.plan.strategy.sort.columns` |
| Cleaning | Delete old file versions and log files | `hoodie.clean.automatic`, `hoodie.cleaner.policy`, `hoodie.cleaner.commits.retained` |
| Archival | Move old timeline instants out of the active timeline | `hoodie.archivelog.folder`, `hoodie.keep.min.commits` |
| Indexing | Maintain the metadata table and stats index | `hoodie.metadata.enable`, `hoodie.metadata.index.column.stats.enable` |
| Rollback | Remove files from failed instants | automatic on failure |
| Savepoint | Pin a version so cleaning cannot delete it | `call create_savepoint(...)` |

Cleaning and metadata configuration:

In [20]:
{
    "hoodie.clean.automatic": "true",
    "hoodie.cleaner.policy": "KEEP_LATEST_COMMITS",
    "hoodie.cleaner.commits.retained": "10",
    "hoodie.metadata.enable": "true",
    "hoodie.metadata.index.column.stats.enable": "true",
    "hoodie.archivelog.folder": "archived",
}

{'hoodie.clean.automatic': 'true',
 'hoodie.cleaner.policy': 'KEEP_LATEST_COMMITS',
 'hoodie.cleaner.commits.retained': '10',
 'hoodie.metadata.enable': 'true',
 'hoodie.metadata.index.column.stats.enable': 'true',
 'hoodie.archivelog.folder': 'archived'}

Run services from SQL procedures (Hudi 0.14+):

In [21]:
try:
    spark
except NameError:
    print("SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.")
else:
    spark.sql("call run_compaction(op => 'schedule', table => 'orders_hudi')")
    spark.sql("call run_clean(table => 'orders_hudi', retain_commits => 10)")
    spark.sql("call run_clustering(table => 'orders_hudi')")
    spark.sql("call show_commits(table => 'orders_hudi', limit => 10)")


SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.


Scheduling guidance:

+ Keep inline services for small tables and simple pipelines; move them async once ingestion is continuous or tables exceed a few TB.
+ Give async services their own Spark job and resources so ingestion SLAs are unaffected.
+ Monitor service lag, not just success: a clean or compaction job that runs but cannot keep up is still an incident.
+ Take a savepoint before risky schema or partition changes so you can roll back to a known instant.

How the services interact. Compaction bounds the number of log files a reader must merge, clustering bounds the number and shape of base files, cleaning bounds the number of file versions, and archival bounds the size of the active timeline. If compaction falls behind, cleaning cannot reclaim as much, and read latency and storage both grow.

A workable schedule. Keep inline cleaning and inline metadata maintenance at all times. Run compaction every few delta commits for small tables and every 15 to 30 minutes for large ones. Run clustering nightly or weekly, off-peak. Take a savepoint before any schema or partition change so you can roll back to a known instant.

Common mistake: disabling a service to make writes faster and never turning it back on. Writes speed up, then reads slow down and storage grows with no bound. Any service you turn off needs a replacement job and an alert on its lag.

## How do you handle CDC with Hudi?

Hudi can ingest CDC events and apply them as upserts/deletes to maintain a current-state table. You typically standardize event ordering and deduplication (record key + precombine field) and monitor lag/compaction. Some setups also keep an append-only log for auditing alongside the current-state table.

Typical ingestion paths:

+ **DeltaStreamer** reads Kafka or a source table, deduplicates by record key and precombine, and upserts into Hudi.
+ **Kafka Connect Hudi sink** writes connector output directly into a Hudi table.
+ **Flink Hudi connector** consumes a changelog stream and supports `changelog` mode for `INSERT`/`UPDATE_BEFORE`/`UPDATE_AFTER`/`DELETE`.
+ **Spark Structured Streaming** with `foreachBatch` calling the Hudi datasource per micro-batch.

In [22]:
try:
    from_json, schema, spark
except NameError:
    print("SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.")
else:
    def upsert_micro_batch(batch_df, batch_id: int) -> None:
        deduped = batch_df.dropDuplicates(["order_id"])

        (
            deduped.write.format("hudi")
            .option("hoodie.table.name", "orders_hudi")
            .option("hoodie.datasource.write.recordkey.field", "order_id")
            .option("hoodie.datasource.write.partitionpath.field", "order_date")
            .option("hoodie.datasource.write.precombine.field", "source_lsn")
            .option("hoodie.datasource.write.operation", "upsert")
            .option("hoodie.datasource.write.table.type", "MERGE_ON_READ")
            .option("hoodie.datasource.write.keygenerator.class", "org.apache.hudi.keygen.SimpleKeyGenerator")
            .mode("append")
            .save("s3://lake/hudi/orders")
        )


    (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", "broker:9092")
        .option("subscribe", "orders.cdc")
        .load()
        .selectExpr("CAST(value AS STRING) AS json")
        .select(from_json("json", schema).alias("r"))
        .select("r.*")
        .writeStream
        .foreachBatch(upsert_micro_batch)
        .option("checkpointLocation", "s3://lake/checkpoints/orders_cdc/")
        .trigger(processingTime="5 minutes")
        .start()
    )


SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.


CDC guidance and failure modes:

+ Order by a source log sequence number (`source_lsn`, `binlog_pos`) rather than wall-clock time; clock skew across shards breaks precombine ordering.
+ Handle tombstones explicitly: a delete event must either use `operation=delete` or carry a null ordering value for soft deletion.
+ Deduplicate within a micro-batch before the upsert, otherwise Hudi resolves duplicates by precombine and you lose the ability to log them.
+ Keep an append-only bronze table of raw CDC events for replay and audit; the Hudi current-state table should be reproducible from it.
+ Watch end-to-end lag: Kafka consumer lag plus commit latency plus compaction backlog.

How ordering stays correct. CDC events from many shards can arrive out of order. Hudi compares records by the precombine field, so a monotonic per-source value such as a binlog position or log sequence number puts the newest event last regardless of arrival order. Wall-clock timestamps from different shards are not comparable because of clock skew.

Tombstones. A delete event can be applied as `operation=delete`, which removes the record, or as an upsert with a null precombine value, which leaves a soft-deleted row. Pick one convention for the whole pipeline and document it, because readers must know whether a missing key means deleted or never seen.

Common mistake: deduplicating only at the sink. Hudi resolves duplicate keys in a batch by precombine, so the extras are silently dropped. Deduplicate explicitly before the upsert if you need to count or log the duplicates.

## What are common operational metrics for Hudi tables?

Common metrics include commit/compaction backlog, file counts per partition, average file size, write latency, failed commits, and query scan metrics. Operational alerts often focus on increasing small files and compaction lag.

Concrete signals to instrument:

+ **Commit duration and rate**: time from batch start to commit on the timeline; alert on p95 regression.
+ **Compaction lag**: number of delta commits since the last compaction, and pending compaction instants. This is the single most important MOR metric.
+ **Clean backlog**: number of file versions beyond retention, and archive timeline growth.
+ **File metrics**: file count per partition, average and p95 file size, small-file ratio, and file-group count.
+ **Write amplification**: bytes read versus bytes written per upsert batch; a ratio near 10x means the index or partition strategy is wrong.
+ **Failed and rolled-back instants**: any rollback deserves investigation.
+ **Metadata table health**: metadata table size and files index freshness.
+ **Read metrics**: scan bytes, snapshot versus read-optimized latency, and query planning time (which improves with the metadata table).

Enable metrics reporters:

In [23]:
{
    "hoodie.metrics.on": "true",
    "hoodie.metrics.reporter.type": "PROMETHEUS",
    "hoodie.metrics.pushgateway.host": "pushgateway.internal",
    "hoodie.metrics.pushgateway.port": "9091",
    "hoodie.metrics.executor.enable": "true",
}

{'hoodie.metrics.on': 'true',
 'hoodie.metrics.reporter.type': 'PROMETHEUS',
 'hoodie.metrics.pushgateway.host': 'pushgateway.internal',
 'hoodie.metrics.pushgateway.port': '9091',
 'hoodie.metrics.executor.enable': 'true'}

Inspect the timeline and file layout directly:

In [24]:
try:
    spark
except NameError:
    print("SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.")
else:
    spark.sql("call show_commits(table => 'orders_hudi', limit => 20)").show(truncate=False)
    spark.sql("call show_fsview_all(table => 'orders_hudi')").show(truncate=False)

    files = (
        spark.read.format("hudi")
        .load("s3://lake/hudi/orders")
        .select("_hoodie_partition_path", "_hoodie_file_name")
        .groupBy("_hoodie_partition_path")
        .count()
    )


SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.


Alerting that works in practice: page on compaction lag exceeding SLA, page on failed commits, and warn on a rising small-file ratio or a growing write-amplification ratio.

Concrete thresholds worth alerting on. Page when compaction lag exceeds the table's SLA window, for example more than 10 delta commits on a MOR table. Warn when the small-file ratio rises above about 20% of files in a partition, or when the average file size drops below half the target. Page on any rollback or failed commit.

Worked example. A MOR table ingests every five minutes with a 15-minute freshness target. `hoodie.compact.inline.max.delta.commits=5` means compaction should follow within about 25 minutes. If the pending delta commits reach 20, snapshot reads merge four times more data than intended, and the alert should fire well before that point.

Common mistake: monitoring only job success. A compaction or clean job that finishes without an error can still be losing ground. Measure lag and backlog, not just exit status.

## When would you choose Hudi vs Iceberg vs Delta?

Hudi is often chosen for CDC-heavy and upsert/delete workloads with near-real-time ingestion requirements. Iceberg is often chosen for engine-agnostic open tables with strong metadata and evolution features. Delta is often chosen when you are aligned with the Delta ecosystem and its tooling. The best choice depends on engines, governance, and operational capabilities.

| Dimension | Hudi | Iceberg | Delta Lake |
|---|---|---|---|
| Primary strength | Upserts, incremental pull, table services | Open multi-engine standard, hidden partitioning | Delta/Databricks ecosystem maturity |
| Update model | Indexed upsert; COW or MOR | Merge-on-read deletes; copy-on-write rewrite | Copy-on-write plus deletion vectors |
| Incremental read | First-class incremental query | Incremental append via snapshots/changelog | Change Data Feed |
| Partition evolution | Clustering, limited spec change | Strong partition evolution | Limited; requires rewrite |
| Streaming ingestion | Strong (DeltaStreamer, Flink, Connect) | Good (Flink, Spark) | Strong within Databricks |
| Engine interoperability | Good, but logs need Hudi-aware readers | Broadest | Broad, Iceberg via UniForm |
| Operational burden | Table services (compaction/clustering) to manage | Maintenance (compaction, expire snapshots) | Maintenance/optimize jobs |

Choose Hudi when the pipeline is defined by continuous CDC and the requirement is "current state fast, plus what changed since last run." Choose Iceberg when the table must be shared across many engines with the least coupling to one vendor. Choose Delta when the platform is Databricks-first and the team wants the most integrated tooling.

Interview answer: do not declare a universal winner. Tie the choice to update patterns, read freshness, engine mix, governance, and who will operate the table services. It is common and defensible to run Hudi for bronze ingestion and Iceberg or Delta for the curated layer.

Worked selection. A team ingests 30 operational databases over Debezium into one lake and must expose current state plus a change feed, on Spark and Trino. Hudi fits the ingestion layer because the index handles high-volume upserts and the incremental query is a first-class read mode. The curated silver layer is written as Iceberg because several engines must read it and the team wants a neutral open standard.

When to invert the choice. If the whole platform is Databricks and the team wants the tightest tooling, Delta is the pragmatic pick. If reads outnumber writes heavily and there are no record-level updates, plain Parquet or Iceberg with copy-on-write is simpler and cheaper to operate than running Hudi table services.

Common mistake: defending a single format as universally best. The honest answer ties the choice to update patterns, read freshness, engine mix, and who operates the table services. A hybrid Hudi-for-ingestion, Iceberg-or-Delta-for-serving layout is common and defensible.

## Real world use cases for Apache Hudi

### 1. E-commerce order events with late-arriving updates

Business context: a marketplace needs a near-real-time order table that reflects cancellations, refunds, and late-arriving status events. Volume: ~40M order events/day (~120 GB/day), peak 15k events/s, freshness target 5 minutes.

```text
Kafka orders.cdc -> Spark Structured Streaming (5 min micro-batch)
  -> Hudi MOR table, record key order_id, precombine source_lsn
  -> read_optimized for BI dashboards, incremental pull for the silver layer
  -> compaction every 5 delta commits, clustering nightly
```

In [25]:
{
    "hoodie.table.name": "orders_hudi",
    "hoodie.datasource.write.recordkey.field": "order_id",
    "hoodie.datasource.write.partitionpath.field": "order_date",
    "hoodie.datasource.write.precombine.field": "source_lsn",
    "hoodie.datasource.write.table.type": "MERGE_ON_READ",
    "hoodie.datasource.write.operation": "upsert",
    "hoodie.compact.inline": "true",
    "hoodie.compact.inline.max.delta.commits": "5",
    "hoodie.parquet.max.file.size": "134217728",
}

{'hoodie.table.name': 'orders_hudi',
 'hoodie.datasource.write.recordkey.field': 'order_id',
 'hoodie.datasource.write.partitionpath.field': 'order_date',
 'hoodie.datasource.write.precombine.field': 'source_lsn',
 'hoodie.datasource.write.table.type': 'MERGE_ON_READ',
 'hoodie.datasource.write.operation': 'upsert',
 'hoodie.compact.inline': 'true',
 'hoodie.compact.inline.max.delta.commits': '5',
 'hoodie.parquet.max.file.size': '134217728'}

Why this design: MOR absorbs the streaming write rate without rewriting Parquet on every event, `source_lsn` makes out-of-order events deterministic, and read-optimized queries keep dashboards fast between compactions. The incremental pull feeds the silver layer without rescanning history.

### 2. Fintech ledger with soft deletes and audit

Business context: a payments ledger must reflect reversals and account closures while retaining an auditable record. Volume: ~8M ledger entries/day, ~2 KB each, strict correctness.

```text
CDC (Debezium) -> bronze raw CDC table (append-only, immutable)
                -> Hudi COW ledger table: hard delete for erasure, soft delete for reversals
                -> incremental pull to regulatory reporting
```

In [26]:
try:
    spark
except NameError:
    print("SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.")
else:
    # Soft delete: null ordering value marks the record as deleted for readers.
    soft_delete = spark.createDataFrame(
        [("txn-9001", "2026-10-04", None)], ["txn_id", "txn_date", "updated_at"]
    )

    (
        soft_delete.write.format("hudi")
        .option("hoodie.table.name", "ledger_hudi")
        .option("hoodie.datasource.write.recordkey.field", "txn_id")
        .option("hoodie.datasource.write.partitionpath.field", "txn_date")
        .option("hoodie.datasource.write.precombine.field", "updated_at")
        .option("hoodie.datasource.write.operation", "upsert")
        .mode("append")
        .save("s3://lake/hudi/ledger")
    )


SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.


Why this design: COW gives the strongest read semantics for audit queries, the bronze CDC log makes the ledger reproducible, and soft deletes preserve history for regulators while `_hoodie_is_deleted`-style filtering hides reversed entries from operational reads. A separate hard-delete job handles GDPR erasure.

### 3. IoT telemetry current-state and incremental analytics

Business context: 200k devices emit readings every 30 seconds; operations needs the latest device state and analytics needs only changed rows. Volume: ~600M events/day (~90 GB/day).

```text
MQTT -> Kafka -> DeltaStreamer -> Hudi MOR device_telemetry
                                    |-> snapshot/read_optimized: current device state
                                    |-> incremental pull (checkpointed) -> silver aggregates
```

```bash
spark-submit --class org.apache.hudi.utilities.deltastreamer.HoodieDeltaStreamer \
  --packages org.apache.hudi:hudi-utilities-bundle_2.12:0.15.0 \
  /opt/hudi/hudi-utilities-bundle.jar \
  --table-type MERGE_ON_READ \
  --source-class org.apache.hudi.utilities.sources.JsonKafkaSource \
  --source-ordering-field event_ts \
  --target-base-path s3://lake/hudi/device_telemetry \
  --target-table device_telemetry \
  --props /opt/hudi/deltastreamer.properties \
  --schemaprovider-class org.apache.hudi.utilities.schema.FilebasedSchemaProvider \
  --continuous
```

Why this design: one Hudi table serves both the low-latency current-state read and the change feed, removing the need to duplicate data. DeltaStreamer handles Kafka offset commit and schema registry integration, and incremental pulls keep the silver aggregation job cheap.

### 4. Multi-tenant SaaS activity lake with tenant isolation

Business context: a B2B SaaS platform ingests customer activity into one governed lake and must serve per-tenant queries with cost attribution. Volume: 500 tenants, ~150M events/day, uneven tenant sizes.

```text
activity API -> Kafka -> Hudi COW table partitioned by event_date,
              record key tenant_id,event_id, precombine ingestion_ts
              GSI-equivalent read patterns via Spark predicate pushdown on tenant_id
```

In [27]:
{
    "hoodie.table.name": "activity_hudi",
    "hoodie.datasource.write.recordkey.field": "tenant_id,event_id",
    "hoodie.datasource.write.keygenerator.class": "org.apache.hudi.keygen.ComplexKeyGenerator",
    "hoodie.datasource.write.partitionpath.field": "event_date",
    "hoodie.datasource.write.precombine.field": "ingestion_ts",
    "hoodie.datasource.write.table.type": "COPY_ON_WRITE",
    "hoodie.metadata.index.column.stats.enable": "true",
}

{'hoodie.table.name': 'activity_hudi',
 'hoodie.datasource.write.recordkey.field': 'tenant_id,event_id',
 'hoodie.datasource.write.keygenerator.class': 'org.apache.hudi.keygen.ComplexKeyGenerator',
 'hoodie.datasource.write.partitionpath.field': 'event_date',
 'hoodie.datasource.write.precombine.field': 'ingestion_ts',
 'hoodie.datasource.write.table.type': 'COPY_ON_WRITE',
 'hoodie.metadata.index.column.stats.enable': 'true'}

Why this design: COW keeps tenant-facing queries simple and fast, composite keys prevent cross-tenant collisions, and date partitioning with column statistics lets Spark prune to a tenant's slices. Column stats in the metadata table keep planning fast despite the file count. Very large tenants can later be split into their own table to avoid skew.

### 5. CDC into a lakehouse bronze layer

Business context: an enterprise replicating 30 operational databases into one lakehouse wants a single upsert-capable bronze table per source before building conformed silver models. Volume: ~1.2B change events/day across sources.

```text
Debezium -> Kafka -> per-source Hudi MOR bronze tables (key = source PK, precombine = lsn)
                  -> incremental pull -> Spark/dbt silver models (Iceberg or Delta)
                  -> compaction + clustering scheduled outside ingestion windows
```

In [28]:
bronze_options = {
    "hoodie.table.name": "crm_accounts_hudi",
    "hoodie.datasource.write.recordkey.field": "account_id",
    "hoodie.datasource.write.precombine.field": "source_lsn",
    "hoodie.datasource.write.table.type": "MERGE_ON_READ",
    "hoodie.datasource.write.operation": "upsert",
    "hoodie.metadata.enable": "true",
    "hoodie.clean.automatic": "true",
    "hoodie.cleaner.commits.retained": "20",
}

Why this design: Hudi absorbs high-volume CDC cheaply at the bronze layer, keeps a replayable current state per source, and exports only changed records downstream. Iceberg or Delta remains the team's choice for the curated layer, which is a normal and interview-defensible hybrid rather than a format war.

How to read these cases. Each one names the workload shape, the table type, and the keys, then explains why that combination holds. In an interview, copy the reasoning rather than the config: state the update rate, the freshness target, the readers, and who owns the table services.

A pattern that repeats across all five cases: Hudi absorbs change at the bronze layer, and the curated layer is rebuilt from an incremental pull. That split keeps the expensive upsert path in one place and keeps downstream models cheap to refresh.

## How do you tune Hudi tables for performance?

Tuning Hudi means balancing four things: write latency, read latency, storage, and operational cost. Almost every knob moves work between them. The order below is the order that pays off in practice: fix file sizing first, then the table type and compaction, then metadata and indexes, then layout.

### Size files before anything else

File size is the single biggest driver of read performance. Too many small files add object store requests, task scheduling overhead, and poor compression. Too few huge files hurt parallelism and make an upsert rewrite more than it needs to.

The two core options are `hoodie.parquet.max.file.size` (target size, default about 120 MB) and `hoodie.parquet.small.file.limit` (files below this are candidates to be filled before a new file is created, default about 100 MB). Set the target near the object store or filesystem block size, commonly 128 MB.

Aim for `hoodie.insert.shuffle.parallelism`, `hoodie.upsert.shuffle.parallelism`, and `hoodie.bulkinsert.shuffle.parallelism` to be roughly `data_size / target_file_size`. A fixed 200 tasks on a 2 GB batch produces many partially filled files.

In [29]:
# Sizing the write parallelism from the batch and the target file size.
TARGET_FILE_BYTES = 128 * 1024 * 1024


def write_parallelism(batch_bytes: int, target_bytes: int = TARGET_FILE_BYTES) -> int:
    """One task per target-sized chunk, with a floor of 1."""
    return max(1, round(batch_bytes / target_bytes))


print(write_parallelism(2 * 1024 ** 3))   # 2 GB batch -> 16 tasks


16


### Choose COW or MOR from the workload, not the fashion

COW pays on every write and saves on every read. MOR saves on writes and pays at read time until compaction runs. With a few percent of rows updated and many readers, COW is usually cheaper end to end. With continuous updates and few readers, MOR absorbs the write rate.

The trade-off is measurable. COW amplification per touched file is roughly read plus write, about 2x. MOR amplification is close to the size of the changed rows, but a snapshot read merges the base file and every log file on top of it. Track bytes read divided by bytes written on the write path, and seconds per snapshot read on the read path.

### Schedule compaction explicitly

Compaction is what keeps a MOR table readable. Schedule it by delta-commit count with `hoodie.compact.inline.max.delta.commits`, or by time with `hoodie.compact.inline.trigger.strategy=TIME_ELAPSED`. Inline compaction keeps operations simple but adds latency to the writer. Async compaction, which is the default in recent versions, runs beside ingestion and is the right choice once ingestion is continuous.

Give compaction its own resources. A compaction reads every log file in the plan and rewrites base files, so it is I/O heavy and can be many GB. Alert on the delta-commit lag, not just on whether the job ran.

In [30]:
try:
    spark
except NameError:
    print("SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.")
else:
    # Schedule compaction and clean from Spark SQL procedures (Hudi 0.14+).
    spark.sql("call run_compaction(op => 'schedule', table => 'orders_hudi')")
    spark.sql("call run_compaction(op => 'run', table => 'orders_hudi')")
    spark.sql("call run_clean(table => 'orders_hudi', retain_commits => 10)")


SKIPPED - needs a Hudi-enabled Spark session and the DataFrames from earlier cells; read on, the markdown explains the flow.


### Use the metadata table for file listing

On cloud object storage, listing a table with hundreds of thousands of files is slow and can hit request rate limits. The metadata table stores the files index, the column stats index, and other indexes as a Hudi table under `.hoodie/metadata`. Readers then resolve the file list from one lookup instead of a full listing.

Enable it with `hoodie.metadata.enable=true` and keep `hoodie.metadata.index.column.stats.enable=true` when queries filter on a few columns. Column stats let the reader skip files by min and max values, which is a large win on selective predicates.

The metadata table is a Hudi table itself, so it needs compaction and cleaning too. Keeping it enabled is effectively mandatory on object storage; the only reason to disable it is a small table on a filesystem where listing is already cheap.

### Pick the index from key cardinality and update volume

`hoodie.index.type` decides how an upsert finds the file groups that hold its keys. Bloom is the general-purpose default. HFile trades extra files for faster repeat lookups on large tables. Simple keeps the key map in driver memory, which is only safe when the key count is small. Bucket removes the lookup entirely but fixes the layout at creation time.

A practical rule:

+ Key count under a few million and the table fits in memory: `SIMPLE` is fastest.
+ Large partitioned table with ordinary upsert traffic: `BLOOM`.
+ Very large table where bloom probing is the bottleneck: `HFILE`.
+ Streaming with a known, stable key space: `BUCKET` with a bucket count that will not need to grow.
+ Keys that move between partitions: a global index, accepting the extra cost.

### Cluster for layout, not as a first fix

Clustering rewrites groups of files into larger, optionally sorted files and commits the swap as a `replacecommit`. It is the right tool for consolidating files after a period of small commits and for sorting on a column that queries filter on.

Use `hoodie.clustering.plan.strategy.small.file.limit` to select candidates and `hoodie.clustering.plan.strategy.target.file.max.bytes` to set the output size. Sort with `hoodie.clustering.plan.strategy.sort.columns` so Parquet min and max statistics prune more row groups.

Clustering is expensive. It rewrites data, creates new file versions, and needs cleaning afterwards. Prefer writing fewer, larger commits and letting auto-sizing handle file sizes; cluster when that is not enough.

### Design the partition path to match queries

A good partition path is a low-cardinality column that queries filter on and that keeps each partition at a healthy size, usually 1 to 5 GB. Daily partitions are the common default. Partitioning by a high-cardinality id, or by hour when volume is low, creates thousands of tiny partitions and undoes every file-sizing improvement.

If no single column is both selective and reasonable in cardinality, keep the partitions coarse and rely on clustering and column stats for pruning inside the partition.

### Tune cleaning and retention against query latency

Cleaning deletes file versions that are older than the retention window. Too little retention and a long-running query or an incremental consumer fails on a missing version. Too much retention and storage grows without bound.

The policy is set with `hoodie.clean.policy` (the older alias `hoodie.cleaner.policy` still works). `KEEP_LATEST_COMMITS` retains the last N commits, set by `hoodie.clean.commits.retained`, which is the policy to use when incremental consumers need a lookback window. `KEEP_LATEST_FILE_VERSIONS` retains N versions per file group, set by `hoodie.clean.fileversions.retained`, and is simpler when there is no time-based lookback.

A safe rule: retain at least `longest_query_minutes / commit_interval_minutes` commits, rounded up, so the oldest version survives the longest reader. Add margin for the slowest day of the week, and keep `hoodie.clean.automatic=true` so cleaning is never forgotten.

## Hudi exercises

These exercises need only the Python standard library. Each one has a task, a hint, the expected result, and a runnable cell with asserts.

### Try it 1. Choose COW or MOR from a read/write profile

**Task.** Write a function `pick_table_type` that returns `"MERGE_ON_READ"` when the freshness target is 15 minutes or less and there are fewer than 5 readers, otherwise `"COPY_ON_WRITE"`. Apply it to a streaming profile (5-minute freshness, 1 reader) and a BI profile (60-minute freshness, 40 readers).

**Hint.** `freshness_minutes <= 15 and readers < 5` is the MOR condition.

**Expected result.** `MERGE_ON_READ` for the stream, `COPY_ON_WRITE` for BI.

In [31]:
def pick_table_type(freshness_minutes: int, readers: int) -> str:
    if freshness_minutes <= 15 and readers < 5:
        return "MERGE_ON_READ"
    return "COPY_ON_WRITE"


stream = pick_table_type(freshness_minutes=5, readers=1)
bi = pick_table_type(freshness_minutes=60, readers=40)

print("streaming profile:", stream)
print("BI profile:", bi)

assert stream == "MERGE_ON_READ"
assert bi == "COPY_ON_WRITE"


streaming profile: MERGE_ON_READ
BI profile: COPY_ON_WRITE


### Try it 2. Size files from ingestion rate and target file size

**Task.** A stream ingests 120 GB per day in five-minute micro-batches. Compute the bytes per batch and the number of 128 MiB target files the writer should aim for. Use 1 GB = 1024 MiB and `math.ceil`.

**Hint.** There are 24 * 60 / 5 = 288 five-minute batches per day. Files per batch is `ceil(batch_mib / 128)`.

**Expected result.** About 426.7 MiB per batch and 4 files per batch.

In [32]:
import math

TARGET_MIB = 128
daily_mib = 120 * 1024
batches_per_day = (24 * 60) // 5
batch_mib = daily_mib / batches_per_day
files_per_batch = math.ceil(batch_mib / TARGET_MIB)

print(f"batches per day = {batches_per_day}")
print(f"batch size = {batch_mib:.1f} MiB")
print(f"target files per batch = {files_per_batch}")

assert batches_per_day == 288
assert round(batch_mib, 1) == 426.7
assert files_per_batch == 4


batches per day = 288
batch size = 426.7 MiB
target files per batch = 4


### Try it 3. Compute the compaction trigger from delta-commit count

**Task.** A MOR table ran 4 delta commits after its last completed compaction. With `hoodie.compact.inline.max.delta.commits=5`, the trigger fires when pending delta commits reach 5. Return whether compaction should be scheduled now, and the number of commits still needed.

**Hint.** `should_compact = pending >= threshold` and `remaining = max(0, threshold - pending)`.

**Expected result.** Not yet, 1 commit remaining.

In [33]:
threshold = 5
pending = 4

should_compact = pending >= threshold
remaining = max(0, threshold - pending)

print("schedule compaction now:", should_compact)
print("delta commits remaining:", remaining)

assert should_compact is False
assert remaining == 1
assert (4 + remaining) >= threshold


schedule compaction now: False
delta commits remaining: 1


### Try it 4. Estimate the metadata-table benefit from file counts

**Task.** A reader lists 200,000 files directly, at 2 ms per list request, and compares that with the metadata table, which resolves the same listing in 50 ms. Compute both times, the speed-up factor, and whether the metadata table saves more than 5 minutes of planning time.

**Hint.** `direct_seconds = file_count * ms_per_call / 1000`. The factor is `direct_seconds / metadata_seconds`.

**Expected result.** 400.0 s direct, 0.05 s with metadata, 8000x faster, planning saved.

In [34]:
file_count = 200_000
ms_per_list_call = 2
metadata_seconds = 0.05

direct_seconds = file_count * ms_per_list_call / 1000
speedup = direct_seconds / metadata_seconds
saves_minutes = (direct_seconds - metadata_seconds) > 5 * 60

print(f"direct listing = {direct_seconds:.1f} s")
print(f"with metadata table = {metadata_seconds:.2f} s")
print(f"speed-up = {speedup:.0f}x")
print("saves over 5 minutes:", saves_minutes)

assert direct_seconds == 400.0
assert speedup == 8000.0
assert saves_minutes is True


direct listing = 400.0 s
with metadata table = 0.05 s
speed-up = 8000x
saves over 5 minutes: True


### Try it 5. Pick an index type from key cardinality

**Task.** Write `pick_index(table_keys, partitioned, streaming, keys_move_partitions)` that returns the Hudi index type. Rules: a global index when keys move partitions, `BUCKET` for streaming with a stable key space, `SIMPLE` when the table has at most 5 million keys, `HFILE` for a large partitioned table, and `BLOOM` otherwise.

**Hint.** Check the rules in priority order and return the first match.

**Expected result.** `GLOBAL_BLOOM`, `BUCKET`, `SIMPLE`, `HFILE`, `BLOOM`.

In [35]:
def pick_index(
    table_keys: int,
    partitioned: bool,
    streaming: bool,
    keys_move_partitions: bool,
) -> str:
    if keys_move_partitions:
        return "GLOBAL_BLOOM"
    if streaming:
        return "BUCKET"
    if table_keys <= 5_000_000:
        return "SIMPLE"
    if partitioned:
        return "HFILE"
    return "BLOOM"


cases = [
    (pick_index(50_000_000, True, True, True), "GLOBAL_BLOOM"),
    (pick_index(50_000_000, True, True, False), "BUCKET"),
    (pick_index(1_000_000, False, False, False), "SIMPLE"),
    (pick_index(500_000_000, True, False, False), "HFILE"),
    (pick_index(500_000_000, False, False, False), "BLOOM"),
]

for got, _ in cases:
    print(got)

for got, expected in cases:
    assert got == expected


GLOBAL_BLOOM
BUCKET
SIMPLE
HFILE
BLOOM


### Try it 6. Plan cleaning retention against query latency

**Task.** The longest query against a table takes 5 hours. Commits happen every 30 minutes. Compute the minimum number of commits to retain under `KEEP_LATEST_COMMITS` so the oldest file version survives the longest reader, then add a 20% safety margin and round up.

**Hint.** `minimum = ceil(longest_query_minutes / commit_interval_minutes)`, then `retained = ceil(minimum * 1.2)`.

**Expected result.** 10 commits minimum, 12 with margin.

In [36]:
import math

longest_query_minutes = 5 * 60
commit_interval_minutes = 30

minimum = math.ceil(longest_query_minutes / commit_interval_minutes)
retained = math.ceil(minimum * 1.2)

print("minimum commits to retain:", minimum)
print("with 20% margin:", retained)

assert minimum == 10
assert retained == 12
assert retained >= minimum


minimum commits to retain: 10
with 20% margin: 12


### Try it 7. Estimate file count after clustering

**Task.** A partition holds 8,000 small files totalling 480 GiB. Clustering targets 1 GiB files. Estimate the file count after clustering, the reduction factor, and check that the new count is at least 1.

**Hint.** `after = max(1, math.ceil(total_gib / target_gib))`. The factor is `before / after`.

**Expected result.** 480 files, 16.7x fewer.

In [37]:
import math

files_before = 8_000
total_gib = 480
target_gib = 1

files_after = max(1, math.ceil(total_gib / target_gib))
reduction = files_before / files_after

print("files before:", files_before)
print("files after:", files_after)
print(f"reduction = {reduction:.1f}x")

assert files_after == 480
assert round(reduction, 1) == 16.7
assert files_after >= 1


files before: 8000
files after: 480
reduction = 16.7x
